# Previsão de série temporal: o protocolo vem antes do modelo

Prever o futuro é diferente de prever o presente. Numa tabela comum, a linha do tempo não existe, as
linhas são independentes e a divisão pode ser sorteada. Numa série temporal, existe **ordem**, as linhas
vizinhas são parecidas, e o que você quer prever é justamente o que ainda não aconteceu.

Isso muda cinco decisões, e este notebook trata das cinco **antes** de treinar qualquer modelo
sofisticado:

1. a **divisão**: o teste é o futuro, e nada do futuro entra no treino;
2. os **baselines**: ingênuo, sazonal e média móvel, que precisam ser batidos antes de qualquer coisa;
3. o **horizonte**: prever 1 passo à frente não é prever 14;
4. o **backtesting**: uma origem só também mente, então a avaliação avança no tempo;
5. as **métricas**: MAE, RMSE, MAPE (e o problema dela), sMAPE e MASE.

A série deste notebook tem componentes **declarados**: tendência, sazonalidade semanal, sazonalidade
anual, ruído e um choque. Assim dá para julgar se o modelo recuperou o que estava lá.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor

sns.set_theme(context="notebook", style="whitegrid", palette="colorblind",
              rc={"figure.figsize": (11, 4)})

RANDOM_STATE = 42


def serie_com_componentes(n_dias=760, seed=RANDOM_STATE):
    """Série diária com verdade declarada: tendência, semana, ano, ruído e um choque."""
    rng = np.random.default_rng(seed)
    t = np.arange(n_dias)
    tendencia = 100 + 0.06 * t
    sazonalidade_semanal = 6.0 * np.sin(2 * np.pi * t / 7)
    sazonalidade_anual = 12.0 * np.sin(2 * np.pi * t / 365)
    ruido = rng.normal(0, 3.0, n_dias)
    choque = np.where((t >= 400) & (t < 415), 25.0 * np.exp(-(t - 400) / 4.0), 0.0)
    indices = pd.date_range("2024-01-01", periods=n_dias, freq="D")
    return pd.Series(tendencia + sazonalidade_semanal + sazonalidade_anual + ruido + choque,
                     index=indices, name="vendas")


serie = serie_com_componentes()
print(f"série: {len(serie)} dias, de {serie.index[0].date()} a {serie.index[-1].date()}")
print(f"média {serie.mean():.1f} | mínimo {serie.min():.1f} | máximo {serie.max():.1f}")
print()
print("componentes declarados: tendência de +0,06 por dia, semana de amplitude 6, "
      "ano de amplitude 12, ruído de desvio 3 e um choque de +25 em 15 dias")

serie.plot()
plt.title("a série, com tudo o que foi declarado dentro dela")
plt.ylabel("vendas")
plt.tight_layout()
plt.show()


## O teste é o futuro

Numa tabela comum, treinar com 75% das linhas sorteadas e testar com as outras 25% é honesto. Numa série,
é outra coisa: se você sorteia, o treino fica com linhas **depois** do teste, e o modelo passa a
interpolar entre vizinhos em vez de extrapolar. O número sai alto e não existe no dia da previsão.

A regra é uma: **treino é passado, teste é futuro, e nada do futuro aparece no treino.** O resto do
notebook é o que essa regra implica.

## Os baselines que precisam ser batidos

Antes de qualquer modelo, três previsões burras: repetir o último valor (ingênuo), repetir o mesmo dia da
semana passada (sazonal) e repetir a média da última semana (média móvel). Em previsão de série, esses
três ganham de modelos caros com frequência, e é por isso que eles vêm primeiro.

Uma armadilha de código que vale a linha: ao construir a média móvel, o `shift(1)` tem de vir **antes**
do `rolling`, senão a janela inclui o próprio valor que se quer prever.


In [ ]:
def previsor_ingenuo(treino, horizonte):
    return np.repeat(treino.iloc[-1], horizonte)


def previsor_sazonal(treino, horizonte, periodo=7):
    ciclo = treino.iloc[-periodo:].to_numpy()
    return np.resize(ciclo, horizonte)


def previsor_media_movel(treino, horizonte, janela=7):
    return np.repeat(treino.iloc[-janela:].mean(), horizonte)


def erro_absoluto_medio(observado, previsto):
    return float(np.mean(np.abs(np.asarray(observado) - np.asarray(previsto))))


def rmse(observado, previsto):
    return float(np.sqrt(np.mean((np.asarray(observado) - np.asarray(previsto)) ** 2)))


def mase(observado, previsto, treino, periodo=7):
    """Erro médio dividido pelo erro do sazonal ingênuo no treino: comparável entre séries."""
    escala = float(np.mean(np.abs(treino.diff(periodo).dropna())))
    return erro_absoluto_medio(observado, previsto) / escala


treino, teste = serie.iloc[:-14], serie.iloc[-14:]
print(f"treino: até {treino.index[-1].date()} | teste: {teste.index[0].date()} a "
      f"{teste.index[-1].date()} ({len(teste)} dias)")

linhas = []
for nome, previsor in {"ingênuo (último valor)": previsor_ingenuo,
                       "sazonal (mesmo dia da semana)": previsor_sazonal,
                       "média móvel de 7": previsor_media_movel}.items():
    previsto = previsor(treino, len(teste))
    linhas.append({"baseline": nome, "MAE": erro_absoluto_medio(teste, previsto),
                   "RMSE": rmse(teste, previsto), "MASE": mase(teste, previsto, treino)})

tabela_baselines = pd.DataFrame(linhas).set_index("baseline").round(3)
display(tabela_baselines)


Leia o MASE, que é a coluna mais informativa: ele é o erro dividido pelo erro de um sazonal ingênuo. Um
MASE de 1,0 significa "tão bom quanto repetir a semana passada"; abaixo de 1, significa que o método
melhorou; e acima de 2 começa a ser embaraçoso.

É esse o sarrafo. Um modelo que não bate a repetição do dia da semana passada não merece o tempo que
gasta para treinar.

Repare também em qual dos três ganhou **neste recorte**: o ingênuo simples, com MASE 1,215, à frente do
sazonal (1,606) e da média móvel (1,556). O motivo está nos componentes declarados: a amplitude semanal é 6
e o ruído tem desvio 3, então repetir ontem é um bom palpite. Em série com sazonalidade forte, o sazonal
ganha. **Qual baseline é o mais forte depende de qual componente domina, e isso se mede, não se adivinha.**

## Backtesting: uma origem só também mente

Do mesmo jeito que uma divisão só mente numa tabela comum, uma origem só mente numa série. É preciso
repetir a avaliação ao longo do tempo, e existem duas formas de janela de treino:

- **expansível**: o treino começa sempre no início e vai crescendo;
- **deslizante**: o treino mantém um tamanho fixo e anda para a frente, esquecendo o passado distante.

Qual das duas usar depende do problema: se o comportamento muda com o tempo, a deslizante representa
melhor o que vai acontecer; se não muda, a expansível aproveita mais dado.


In [ ]:
def backtest(serie, previsor, origens=10, passos=14, janela=None):
    """Avaliação com origem deslizante: treina até uma origem e prevê os próximos `passos`.

    Devolve duas coisas: o resumo por origem (a variação entre períodos) e o erro de cada passo
    em cada origem, que é o que permite olhar o horizonte sem misturar tudo numa média só.
    """
    por_origem, erro_por_passo = [], []
    for origem in range(len(serie) - passos * origens, len(serie) - passos, passos):
        treino = serie.iloc[:origem] if janela is None else serie.iloc[origem - janela:origem]
        alvo = serie.iloc[origem:origem + passos]
        previsto = previsor(treino, passos)
        por_origem.append({
            "origem": serie.index[origem].date(),
            "MAE": erro_absoluto_medio(alvo, previsto),
            "MASE": mase(alvo, previsto, treino),
        })
        erro_por_passo.append(np.abs(alvo.to_numpy() - np.asarray(previsto)))
    return pd.DataFrame(por_origem), np.vstack(erro_por_passo)


medidas = {"sazonal": previsor_sazonal, "média móvel de 7": previsor_media_movel}

for nome, previsor in medidas.items():
    resultado, _ = backtest(serie, previsor)
    print(f"{nome:20s} MASE por origem: {[round(v, 2) for v in resultado['MASE']]}")
    print(f"{'':20s} média {resultado['MASE'].mean():.3f} | "
          f"de {resultado['MASE'].min():.3f} a {resultado['MASE'].max():.3f}")
    print()

resultado_sazonal, passos_sazonal = backtest(serie, previsor_sazonal)
sns.boxplot(resultado_sazonal["MASE"], color="lightsteelblue")
sns.stripplot(resultado_sazonal["MASE"], color="black", size=6)
plt.title("MASE por origem do backtesting, mesmo método")
plt.xlabel("MASE")
plt.tight_layout()
plt.show()


O espalhamento entre origens é a razão de existir o backtesting: o mesmo método, na mesma série, dá
resultados diferentes conforme o período que cai no teste. Reportar a média sem mostrar a variação esconde
justamente a informação que diz se o número é confiável. Aqui o sazonal ingênuo vai de MASE 0,89 a 1,41
entre as nove origens.

E tem uma segunda lição nesses números, que só aparece quando se mede mais de uma janela: no recorte de 14
dias do início, o ingênuo simples ganhou dos três; no backtesting de nove origens, quem ganha é o sazonal.
**A ordem dos baselines muda com a janela de avaliação**, e é por isso que uma janela só não decide nada.

Isso é a mesma lição de medir com validação cruzada em tabela comum, com uma diferença: aqui a divisão
**não pode ser sorteada**, ela tem de respeitar a ordem.


### O mesmo modelo, dois protocolos

A regra "o teste é o futuro" fica mais concreta com número. O modelo abaixo é o mesmo nas duas medições:
uma árvore de boosting treinada em lags (ontem, anteontem, a mesma semana passada) e no calendário do dia.
O que muda é só o protocolo de avaliação.


In [ ]:
from sklearn.model_selection import KFold, cross_val_score


def tabela_de_lags(serie, atrasos=(1, 2, 7, 14)):
    """Cada linha é um dia, descrito pelos valores anteriores e pelo calendário dele."""
    tabela = pd.DataFrame({"valor": serie})
    for atraso in atrasos:
        tabela[f"lag_{atraso}"] = tabela["valor"].shift(atraso)
    tabela["dia_da_semana"] = serie.index.dayofweek
    tabela["dia_do_ano"] = serie.index.dayofyear
    return tabela.dropna()


def previsor_boosting(treino, horizonte):
    """Treina nos lags e prevê recursivamente: cada previsão vira dado para a próxima."""
    historico = treino.copy()
    tabela_treino = tabela_de_lags(historico)
    modelo = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
    modelo.fit(tabela_treino.drop(columns="valor"), tabela_treino["valor"])

    previsoes = []
    for _ in range(horizonte):
        proximo_dia = historico.index[-1] + pd.Timedelta(days=1)
        linha = pd.DataFrame([{
            "lag_1": historico.iloc[-1], "lag_2": historico.iloc[-2],
            "lag_7": historico.iloc[-7], "lag_14": historico.iloc[-14],
            "dia_da_semana": proximo_dia.dayofweek, "dia_do_ano": proximo_dia.dayofyear,
        }])
        previsao = float(modelo.predict(linha)[0])
        previsoes.append(previsao)
        historico.loc[proximo_dia] = previsao
    return np.array(previsoes)


tabela_lags = tabela_de_lags(serie)
alvo_lags = tabela_lags.pop("valor")
modelo_lags = HistGradientBoostingRegressor(random_state=RANDOM_STATE)

notas_sorteadas = cross_val_score(modelo_lags, tabela_lags, alvo_lags,
                                  cv=KFold(5, shuffle=True, random_state=RANDOM_STATE),
                                  scoring="neg_mean_absolute_error")
resultado_boosting, passos_boosting = backtest(serie, previsor_boosting)

print("o mesmo modelo, dois protocolos:")
print(f"  divisão sorteada (errada): MAE {-notas_sorteadas.mean():6.3f}")
print(f"  backtesting (certo):       MAE {resultado_boosting['MAE'].mean():6.3f}")
print()
print("no mesmo backtesting:")
print(f"  baseline sazonal ingênuo:  MAE {resultado_sazonal['MAE'].mean():6.3f}")
print(f"  baseline ingênuo simples:  MAE "
      f"{backtest(serie, previsor_ingenuo)[0]['MAE'].mean():6.3f}")


A divisão sorteada dá um número melhor, e é o número falso: com as linhas embaralhadas, todo dia de teste
tem os vizinhos no treino, e o modelo interpola em vez de prever. No backtesting, o modelo só vê passado, e
a nota cai para perto do sazonal ingênuo.

Se a diferença entre os dois protocolos for maior que a diferença entre os métodos que você está
comparando, quem decidiu o resultado foi o protocolo, não o método.

## O horizonte muda tudo, mas não do jeito que se imagina

Prever o próximo dia e prever os próximos quatorze não são a mesma tarefa, e toda medição precisa dizer
**para quantos passos** ela vale. Mas o erro não cresce do mesmo jeito para todo método, e a diferença
tem uma explicação que vale mais que a regra geral.

Existem duas estratégias quando o horizonte passa de um passo:

- **recursiva**: prever um passo, usar a previsão como se fosse dado e prever o próximo;
- **direta**: treinar um modelo por passo, cada um com o seu alvo.

A recursiva acumula erro, porque alimenta o modelo com as próprias previsões. Já um método que apenas copia
um valor lá do passado, como o sazonal ingênuo, não acumula nada: o erro dele depende do passo, não do
caminho até ele.


In [ ]:
passos = np.arange(1, passos_sazonal.shape[1] + 1)

tabela_horizonte = pd.DataFrame({
    "passo à frente": passos,
    "sazonal ingênuo (copia o passado)": passos_sazonal.mean(axis=0),
    "boosting recursivo (acumula)": passos_boosting.mean(axis=0),
}).set_index("passo à frente").round(3)
display(tabela_horizonte)

tabela_horizonte.plot(marker="o")
plt.title("erro por passo à frente, média entre as origens do backtesting")
plt.ylabel("MAE")
plt.xlabel("passos à frente")
plt.tight_layout()
plt.show()

for nome, passos_modelo in {"sazonal ingênuo": passos_sazonal,
                            "boosting recursivo": passos_boosting}.items():
    media = passos_modelo.mean(axis=0)
    print(f"{nome:20s} do passo 1 ao 14: {media[0]:.3f} -> {media[-1]:.3f} | "
          f"faixa {media.min():.3f} a {media.max():.3f}")


Leia a tabela e o gráfico com atenção, porque eles desmentem a frase mais repetida sobre previsão: "o erro
cresce com o horizonte" não vale para todo método.

O sazonal ingênuo vai do passo 1 ao 14 praticamente no mesmo nível, 4,596 para 4,476, com uma faixa que
oscila entre 2,244 e 4,931, e volta a cair nos passos múltiplos de 7. Ele copia um valor do passado, então
não tem como acumular nada.

O boosting recursivo começa igual, 4,668, e termina quase 40% pior, 6,392, porque cada previsão dele entra
como dado para a próxima. O erro que ele acumula é o preço da estratégia recursiva, e a alternativa
**direta**, treinando um modelo por passo, cobra o preço em treinos.

Duas consequências práticas: se você reporta o erro médio de todos os passos, está misturando horizontes
diferentes numa média só; e se o seu método é recursivo, o desempenho no passo 1 não autoriza prometer nada
sobre o passo 14.

## As métricas, e por que existe mais de uma

Nenhuma métrica é neutra: cada uma pune um tipo de erro e ignora outro.

- **MAE**: erro médio em unidade da série; fácil de explicar, e trata erro grande e pequeno igual;
- **RMSE**: pune erro grande mais que o MAE, porque eleva ao quadrado;
- **MAPE**: erro percentual; parece a mais intuitiva e é a mais perigosa, porque **não existe** quando o
  valor real é zero, e distorce a média em série com valores baixos;
- **sMAPE**: uma correção do MAPE, com limite de 200% e problemas próprios;
- **MASE**: erro dividido pelo sazonal ingênuo; é o único comparável entre séries de escalas diferentes.

Testar intervenção com métrica que não distingue é o jeito mais rápido de publicar melhora que não existe.


In [ ]:
def mape(observado, previsto):
    observado, previsto = np.asarray(observado, dtype=float), np.asarray(previsto, dtype=float)
    return float(np.mean(np.abs((observado - previsto) / observado)) * 100)


def smape(observado, previsto):
    observado, previsto = np.asarray(observado, dtype=float), np.asarray(previsto, dtype=float)
    return float(np.mean(2 * np.abs(previsto - observado) / (np.abs(observado) + np.abs(previsto))) * 100)


previsto_sazonal = previsor_sazonal(treino, len(teste))
print("as mesmas previsões, quatro métricas:")
print(f"  MAE   {erro_absoluto_medio(teste, previsto_sazonal):8.3f}")
print(f"  RMSE  {rmse(teste, previsto_sazonal):8.3f}")
print(f"  MAPE  {mape(teste, previsto_sazonal):8.3f}%")
print(f"  sMAPE {smape(teste, previsto_sazonal):8.3f}%")
print(f"  MASE  {mase(teste, previsto_sazonal, treino):8.3f}")
print()
print("e o MAPE no caso em que o real é zero:")
print(f"  observado [0, 5, 10], previsto [1, 5, 9] -> MAPE {mape([0, 5, 10], [1, 5, 9])}")


O MAPE com um zero na conta devolve `inf`, ou seja, a métrica simplesmente não existe para aquele ponto, e
quem usa a média de MAPE em série com valor baixo está somando infinito com normal. É por isso que MASE
existe: ele é o erro dividido por um erro de referência da própria série, então 1,0 tem o mesmo
significado em qualquer série, e zero não quebra nada.

## Como ler isso

1. **a divisão é temporal**: treino é passado, teste é futuro, e nada embaralha;
2. **antes do modelo, os baselines**: ingênuo, sazonal, média móvel, e o MASE deles como sarrafo;
3. **diga o horizonte**: erro sem horizonte não significa nada;
4. **uma origem não é avaliação**: rode o backtesting, mostra a média e a variação;
5. **escolha a métrica pelo problema**: MAE para unidade, RMSE para punir erro grande, MASE para comparar
   séries, MAPE só quando não existe valor perto de zero;
6. antes de acreditar num resultado bom, pergunte se algo do futuro entrou no treino.

## Resumo

Previsão de série temporal é protocolo antes de modelo. Neste notebook: a série tem componentes
declarados e um choque no meio; os três baselines são medidos, e quem ganha muda com a janela de avaliação
(o ingênuo simples leva no recorte de 14 dias, com MASE 1,215, e o sazonal leva no backtesting de nove
origens, com 1,119); o mesmo modelo lido com divisão sorteada dá MAE 3,167 e com backtesting dá 4,480 --
o primeiro número é o falso; o erro do sazonal ingênuo oscila em vez de crescer (4,596 no passo 1, 4,476 no
14), enquanto o boosting recursivo cresce até 6,392 porque consome as próprias previsões; e o MAPE devolve
`inf` com um zero na conta, problema que o MASE não tem.

**Próximo passo:** o notebook de previsores clássicos desta pasta, onde ETS e SARIMAX entram para tentar
bater o sazonal ingênuo.

## Referências

```bibtex
@article{hyndman2006another,
  title   = {Another Look at Measures of Forecast Accuracy},
  author  = {Hyndman, Rob J. and Koehler, Anne B.},
  journal = {International Journal of Forecasting},
  volume  = {22},
  number  = {4},
  pages   = {679--688},
  year    = {2006},
  doi     = {10.1016/j.ijforecast.2006.03.001}
}

@article{makridakis2018m4,
  title   = {The M4 Competition: 100,000 Time Series and 61 Forecasting Methods},
  author  = {Makridakis, Spyros and Spiliotis, Evangelos and Assimakopoulos, Vassilios},
  journal = {International Journal of Forecasting},
  volume  = {36},
  number  = {1},
  pages   = {54--74},
  year    = {2020},
  doi     = {10.1016/j.ijforecast.2019.04.014}
}

@article{bergmeir2012use,
  title   = {On the Use of Cross-Validation for Time Series Predictor Evaluation},
  author  = {Bergmeir, Christoph and Ben{'i}tez, Jos{'e} M.},
  journal = {Information Sciences},
  volume  = {191},
  pages   = {192--213},
  year    = {2012},
  doi     = {10.1016/j.ins.2012.02.016}
}
```
